# RetailEdge Analytics Dashboard
## Exploratory Data Analysis Notebook
**Dataset:** UK Online Retail — FY 2023 (Synthetic)  
**Tools:** Python · Pandas · Matplotlib · Seaborn  

---
This notebook walks through the full EDA pipeline:  
1. Data loading & inspection  
2. Cleaning & validation  
3. Univariate analysis  
4. Revenue trend analysis  
5. Customer analysis & RFM  
6. Product & category performance  
7. Geographic breakdown  
8. Key insights summary  


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Dashboard colour theme
BG, PANEL   = "#1E2130", "#252A3D"
ACCENT1, ACCENT2, ACCENT3 = "#4E9AF1", "#F1714E", "#4EF1A0"
TEXT_MAIN, TEXT_DIM = "#EAEAEA", "#8A8FA8"
PALETTE = [ACCENT1, ACCENT2, ACCENT3, "#C47AF1", "#F1D44E",
           "#4EE0F1", "#F14EA0", "#A0F14E"]

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': PANEL,
    'text.color': TEXT_MAIN, 'axes.labelcolor': TEXT_DIM,
    'xtick.color': TEXT_DIM, 'ytick.color': TEXT_DIM,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.spines.left': False, 'axes.spines.bottom': False,
    'axes.grid': True, 'grid.color': '#2E3450', 'grid.linewidth': 0.6,
})

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 20)

df_raw = pd.read_csv('../data/raw/retail_sales_raw.csv', parse_dates=['InvoiceDate'])
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 2. Data Inspection

In [ ]:
print("=== Data Types ===")
print(df_raw.dtypes)
print("\n=== Null Counts ===")
print(df_raw.isnull().sum())
print("\n=== Basic Stats ===")
df_raw[['Quantity','UnitPrice','LineTotal','CostPrice']].describe()

## 3. Data Cleaning

In [ ]:
df = df_raw.copy()
df = df.drop_duplicates()
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df = df.dropna(subset=['InvoiceNo','InvoiceDate','Country','StockCode'])

# Derived columns
df['Year']        = df['InvoiceDate'].dt.year
df['Month']       = df['InvoiceDate'].dt.month
df['MonthName']   = df['InvoiceDate'].dt.strftime('%b')
df['Quarter']     = df['InvoiceDate'].dt.quarter.map({1:'Q1',2:'Q2',3:'Q3',4:'Q4'})
df['DayOfWeek']   = df['InvoiceDate'].dt.day_name()
df['GrossProfit'] = (df['LineTotal'] - df['CostPrice']).round(2)
df['Margin_pct']  = ((df['GrossProfit'] / df['LineTotal']) * 100).round(2)
df['IsGuest']     = df['CustomerID'] == 'GUEST'

print(f"Clean rows: {len(df):,}")
print(f"Date range: {df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")
print(f"Total Revenue: GBP {df['LineTotal'].sum():,.2f}")
print(f"Total Profit : GBP {df['GrossProfit'].sum():,.2f}")

## 4. Monthly Revenue Trend

In [ ]:
monthly = (df.groupby(['Month','MonthName'])
             .agg(Revenue=('LineTotal','sum'), Profit=('GrossProfit','sum'),
                  Orders=('InvoiceNo','nunique'))
             .reset_index().sort_values('Month'))

months = monthly['MonthName'].tolist()
rev    = (monthly['Revenue'] / 1000).tolist()
profit = (monthly['Profit']  / 1000).tolist()
x = np.arange(len(months))

fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG)
ax.fill_between(x, rev,    alpha=0.15, color=ACCENT1)
ax.fill_between(x, profit, alpha=0.15, color=ACCENT3)
ax.plot(x, rev,    color=ACCENT1, lw=2.5, marker='o', ms=6,
        mfc=BG, mew=2, label='Revenue')
ax.plot(x, profit, color=ACCENT3, lw=2.5, marker='s', ms=6,
        mfc=BG, mew=2, label='Gross Profit')
ax.set_xticks(x); ax.set_xticklabels(months)
ax.set_title('Monthly Revenue vs Gross Profit (GBP thousands)',
             color=TEXT_MAIN, fontsize=13, fontweight='bold', pad=10, loc='left')
ax.set_ylabel('GBP (thousands)')
ax.legend(facecolor=PANEL, edgecolor='#2E3450', labelcolor=TEXT_DIM)
plt.tight_layout(); plt.show()
print(monthly[['MonthName','Revenue','Profit','Orders']].to_string(index=False))

## 5. Top 10 Customers

In [ ]:
top_cust = (df[df['IsGuest']==False]
               .groupby(['CustomerID','CustomerName'])
               .agg(Revenue=('LineTotal','sum'),
                    Orders=('InvoiceNo','nunique'),
                    Profit=('GrossProfit','sum'))
               .reset_index()
               .sort_values('Revenue', ascending=False)
               .head(10))

fig, ax = plt.subplots(figsize=(10, 5.5), facecolor=BG)
names = top_cust['CustomerName'].tolist()
vals  = (top_cust['Revenue'] / 1000).tolist()
colors = [ACCENT2 if i==0 else ACCENT1 for i in range(len(names))]
bars = ax.barh(names[::-1], vals[::-1], color=colors[::-1], height=0.62, zorder=3)
for bar, val in zip(bars, vals[::-1]):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f'GBP {val:.1f}k', va='center', color=TEXT_DIM, fontsize=9)
ax.set_title('Top 10 Customers by Revenue', color=TEXT_MAIN, fontsize=13,
             fontweight='bold', pad=10, loc='left')
ax.set_xlabel('Revenue (GBP thousands)')
ax.set_xlim(0, max(vals)*1.18)
plt.tight_layout(); plt.show()
top_cust[['CustomerName','Revenue','Orders','Profit']].head(10)

## 6. Revenue by Country

In [ ]:
country_rev = (df[df['Country']!='United Kingdom']
                  .groupby('Country')
                  .agg(Revenue=('LineTotal','sum'),
                       Orders=('InvoiceNo','nunique'),
                       Profit=('GrossProfit','sum'))
                  .reset_index()
                  .sort_values('Revenue', ascending=False)
                  .head(10))

fig, ax = plt.subplots(figsize=(11, 4.5), facecolor=BG)
vals = (country_rev['Revenue']/1000).tolist()
colors = [ACCENT2 if i==0 else '#5B8CF5' for i in range(len(vals))]
bars = ax.bar(country_rev['Country'], vals, color=colors, width=0.62, zorder=3)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'GBP {val:.0f}k', ha='center', color=TEXT_DIM, fontsize=8.5)
ax.set_title('Top 10 International Markets (excl. UK)', color=TEXT_MAIN,
             fontsize=13, fontweight='bold', pad=10, loc='left')
ax.set_ylabel('Revenue (GBP thousands)')
ax.tick_params(axis='x', rotation=30)
ax.set_ylim(0, max(vals)*1.18)
plt.tight_layout(); plt.show()

## 7. Product Category Analysis

In [ ]:
cat = (df.groupby('Category')
         .agg(Revenue=('LineTotal','sum'), Units=('Quantity','sum'),
              Profit=('GrossProfit','sum'))
         .reset_index().sort_values('Revenue', ascending=False))
cat['Margin%'] = ((cat['Profit']/cat['Revenue'])*100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)

# Donut
vals, labels = cat['Revenue'].tolist(), cat['Category'].tolist()
wedges, _, autotexts = axes[0].pie(
    vals, autopct='%1.1f%%', colors=PALETTE[:len(labels)],
    wedgeprops=dict(width=0.55, edgecolor=BG, linewidth=2),
    startangle=90, pctdistance=0.78)
for at in autotexts:
    at.set_color(BG); at.set_fontsize(8); at.set_fontweight('bold')
axes[0].legend(wedges, labels, loc='lower center', bbox_to_anchor=(0.5,-0.15),
               ncol=2, facecolor=PANEL, edgecolor='#2E3450',
               labelcolor=TEXT_DIM, fontsize=8)
axes[0].set_title('Revenue Share by Category', color=TEXT_MAIN,
                  fontsize=12, fontweight='bold', pad=10)
axes[0].text(0,0,f"GBP {sum(vals)/1000:.0f}k", ha='center',
             va='center', color=TEXT_MAIN, fontsize=12, fontweight='bold')

# Margin bar
axes[1].barh(cat['Category'][::-1], cat['Margin%'][::-1],
             color=PALETTE[:len(cat)][::-1], height=0.6, zorder=3)
axes[1].set_title('Gross Margin % by Category', color=TEXT_MAIN,
                  fontsize=12, fontweight='bold', pad=10, loc='left')
axes[1].set_xlabel('Margin (%)')
plt.tight_layout(); plt.show()
cat

## 8. RFM Customer Segmentation

In [ ]:
ref = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = (df[df['IsGuest']==False]
       .groupby('CustomerID')
       .agg(Recency=('InvoiceDate', lambda x:(ref-x.max()).days),
            Frequency=('InvoiceNo','nunique'),
            Monetary=('LineTotal','sum'))
       .reset_index())

rfm['R'] = pd.qcut(rfm['Recency'],   q=4, labels=[4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['Frequency'], q=4, labels=[1,2,3,4]).astype(int)
rfm['M'] = pd.qcut(rfm['Monetary'],  q=4, labels=[1,2,3,4]).astype(int)
rfm['Score'] = rfm['R'] + rfm['F'] + rfm['M']

def seg(s):
    if s>=10: return 'Champions'
    elif s>=8: return 'Loyal Customers'
    elif s>=6: return 'Potential Loyalists'
    elif s>=4: return 'At Risk'
    else: return 'Lost'

rfm['Segment'] = rfm['Score'].apply(seg)
seg_order = ['Champions','Loyal Customers','Potential Loyalists','At Risk','Lost']
seg_colors= [ACCENT3, ACCENT1, '#C47AF1', ACCENT2, '#888']
counts = rfm['Segment'].value_counts().reindex(seg_order).fillna(0)

fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
bars = ax.bar(counts.index, counts.values, color=seg_colors, width=0.6, zorder=3)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            str(int(val)), ha='center', color=TEXT_DIM, fontsize=11, fontweight='bold')
ax.set_title('RFM Customer Segmentation', color=TEXT_MAIN, fontsize=13,
             fontweight='bold', pad=10, loc='left')
ax.set_ylabel('Number of Customers')
plt.tight_layout(); plt.show()
rfm.groupby('Segment')[['Recency','Frequency','Monetary']].mean().round(2)

## 9. Key Insights Summary

In [ ]:
total_rev   = df['LineTotal'].sum()
total_prof  = df['GrossProfit'].sum()
top_country = df[df['Country']!='United Kingdom'].groupby('Country')['LineTotal'].sum().idxmax()
top_customer= df[df['IsGuest']==False].groupby('CustomerName')['LineTotal'].sum().idxmax()
top_cat     = df.groupby('Category')['LineTotal'].sum().idxmax()
best_month  = df.groupby('MonthName')['LineTotal'].sum().idxmax()

print("=" * 55)
print("  RETAILEDGE ANALYTICS — KEY INSIGHTS SUMMARY")
print("=" * 55)
print(f"  Total Revenue       : GBP {total_rev:>12,.2f}")
print(f"  Total Gross Profit  : GBP {total_prof:>12,.2f}")
print(f"  Overall Margin      : {(total_prof/total_rev*100):.1f}%")
print(f"  Top International   : {top_country}")
print(f"  Top Customer        : {top_customer}")
print(f"  Top Category        : {top_cat}")
print(f"  Peak Revenue Month  : {best_month}")
print("=" * 55)

## 10. Next Steps
- Connect cleaned CSV to **Power BI** for interactive dashboard
- Schedule `data_cleaning.py` as a cron job for automated refreshes
- Load into PostgreSQL using `schema.sql` and run `queries.sql` for live reporting
- Extend RFM model with churn probability scoring
